In [12]:
import pandas as pd
import re
import json

In [14]:

input_file = "datasets/statement.txt"

with open(input_file, "r", encoding="utf-8") as file:
    text = file.read()


text = re.sub(
    r'Page:\s*\d+',
    '',
    text,
    flags=re.IGNORECASE
)

# Remove repeated bank statement headers
text = re.sub(
    r'GOBINDA PRASAD SUBEDI.*?From\s+\d{2}-\d{2}-\d{4}\s+to\s+\d{2}-\d{2}-\d{4}',
    '',
    text,
    flags=re.DOTALL | re.IGNORECASE
)

# Remove table header lines
text = re.sub(
    r'-{20,}',
    '',
    text
)

text = re.sub(
    r'Date\s+Value Date\s+Description\s+Debit\s+Credit\s+Balance',
    '',
    text,
    flags=re.IGNORECASE
)


pattern = re.compile(
    r'(?ms)'
    r'^\s*'
    r'(\d{2}-[A-Za-z]{3})'          # Date
    r'\s+'
    r'(\d{2}[A-Za-z]{3}\d{2})'     # Value Date
    r'\s+'
    r'(.*?)'
    r'(?='
        r'^\s*\d{2}-[A-Za-z]{3}\s+\d{2}[A-Za-z]{3}\d{2}'
        r'|\Z'
    ')'
)

transactions = pattern.findall(text)

print("Transactions found:", len(transactions))


rows = []

for date, value_date, block in transactions:

    amounts = re.findall(
        r'\d[\d,]*\.\d{2}',
        block
    )

    # Skip if no amount exists
    if not amounts:
        continue


    balance = amounts[-1]

    transaction_amount = None

    if len(amounts) >= 2:
        transaction_amount = amounts[-2]


  
    debit = None
    credit = None

    lines = block.splitlines()

    for line in lines:

        # Find monetary amount in this line
        match = re.search(
            r'\d[\d,]*\.\d{2}',
            line
        )

        if not match:
            continue

        amount = match.group()

        # Ignore the balance if this is the last amount
        if amount == balance and len(amounts) > 1:
            # There can be cases where the balance appears
            # on another line, so don't immediately skip.
            pass

        # Position of amount in the original text line
        position = match.start()

        # ----------------------------------------------------
        # Debit is normally positioned around the Debit column
        # Credit is positioned further to the right
        # ----------------------------------------------------

        if amount == transaction_amount:

            if position < 65:
                debit = transaction_amount
            else:
                credit = transaction_amount

            break


    # ========================================================
    # 6. CLEAN DESCRIPTION
    # ========================================================

    description = block

    # Remove all monetary values
    description = re.sub(
        r'\d[\d,]*\.\d{2}',
        ' ',
        description
    )

    # Remove excessive spaces/newlines
    description = re.sub(
        r'\s+',
        ' ',
        description
    )

    description = description.strip()


    # ========================================================
    # 7. ADD ROW
    # ========================================================

    rows.append({
        "Date": date,
        "Value Date": value_date,
        "Description": description,
        "Debit": debit,
        "Credit": credit,
        "Balance": balance
    })


# ============================================================
# 8. CREATE PANDAS DATAFRAME
# ============================================================

df = pd.DataFrame(rows)


# ============================================================
# 9. CLEAN NUMERIC COLUMNS
# ============================================================

for column in ["Debit", "Credit", "Balance"]:

    df[column] = (
        df[column]
        .astype(str)
        .replace("None", "")
        .replace("nan", "")
        .str.replace(",", "", regex=False)
    )

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 10. CLEAN DESCRIPTION FURTHER
# ============================================================

df["Description"] = (
    df["Description"]
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)


# ============================================================
# 11. DISPLAY RESULT
# ============================================================

print("\nClean Transaction Data:")
print("=" * 100)

print(df.to_string(index=False))


# ============================================================
# 12. SAVE AS CSV
# ============================================================

csv_file = "clean_statement.csv"

df.to_csv(
    csv_file,
    index=False
)

print("\nCSV saved as:", csv_file)


# ============================================================
# 13. SAVE AS JSON
# ============================================================

json_file = "clean_statement.json"

# Convert NaN to None
json_data = df.where(
    pd.notnull(df),
    None
).to_dict(orient="records")

with open(
    json_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        json_data,
        file,
        indent=4,
        ensure_ascii=False
    )

print("JSON saved as:", json_file)


# ============================================================
# 14. OPTIONAL: SAVE AS EXCEL
# ============================================================

excel_file = "clean_statement.xlsx"

df.to_excel(
    excel_file,
    index=False
)

print("Excel saved as:", excel_file)


# ============================================================
# DONE
# ============================================================

print("\nProcessing completed successfully!")

Transactions found: 395

Clean Transaction Data:
  Date Value Date                                                                                                      Description       Debit  Credit     Balance
18-Jul    18Jul25                              9855071250/bankXPMob,GOBINDA PRASAD SUBEDI,pay/ET TO 042 00524200087 M69527517 0420     1000.00     NaN    65263.35
18-Jul    18Jul25                              9855071250/bankXPMob,GOBINDA PRASAD SUBEDI,pay/ET TO 042 00501205275 M69527594 0420     1600.00     NaN    63663.35
18-Jul    18Jul25                                                                       4813670002486602 TRACE 13009639/2 EB042001    10000.00     NaN    53663.35
19-Jul    19Jul25                              9855071250/bankXPMob,GOBINDA PRASAD SUBEDI,kharch TO 001 00501227190 M69700061 0420     5000.00     NaN    48663.35
20-Jul    20Jul25                                                               HIMALAYAN BANK FPO-ASBA CHRG 1301080001248131 0000      

In [20]:
df=pd.read_csv("clean_statement.csv")

df = df[
    df["Description"]
    .str.contains("TARA NATH POKHAREL", case=False, na=False)
]
df

for column in ["Debit", "Credit", "Balance"]:

    df[column] = (
        df[column]
        .astype(str)
        .replace("None", "")
        .replace("nan", "")
        .str.replace(",", "", regex=False)
    )

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 11. DISPLAY
# ============================================================

print(df.to_string(index=False))
result = df[
    df["Description"]
    .str.contains("TARA NATH POKHAREL", case=False, na=False)
]

print(result)

  Date Value Date                                                                  Description   Debit  Credit   Balance
25-Apr    25Apr26 TARA NATH POKHAREL/NABIL Cip s-(1001/04200501202549/Hou C IPS/640635158 0010 45000.0     NaN 771258.08
18-Jun    18Jun26 TARA NATH POKHAREL/NABIL Cip s-(1001/04200501202549/Hou C IPS/669185794 0010 45000.0     NaN 240108.58
       Date Value Date                                        Description  \
289  25-Apr    25Apr26  TARA NATH POKHAREL/NABIL Cip s-(1001/042005012...   
365  18-Jun    18Jun26  TARA NATH POKHAREL/NABIL Cip s-(1001/042005012...   

       Debit  Credit    Balance  
289  45000.0     NaN  771258.08  
365  45000.0     NaN  240108.58  
